# Composing Subagents

So far, each tutorial has centered on one agent. That is useful for learning the mechanics, but real workflows often split naturally into smaller jobs.

In NOOA, a subagent is just another `Agent` instance. The parent creates it, passes explicit inputs, awaits one of its methods, and combines the result with normal Python.

This notebook keeps the example small and concrete: a weekend travel planner. We'll build a `WeekendPlannerAgent` with two children:

- **`TripScoutAgent`** uses `PredictStrategy` to extract trip preferences from a messy traveler request.
- **`ItineraryBuilderAgent`** uses the default `CodeActStrategy` to inspect a small city guide through helper methods and write the plan.

The thing to watch is the composition pattern: the parent is a plain Python orchestrator, and each child owns one focused LLM task.


## Prerequisites

Run the install cell below before the setup cell.

The setup cell below lists several providers. Uncomment the one you want to use. For hosted providers, replace `"your-api-key"` with a real key. NOOA is also compatible with **local inference** (Ollama, vLLM, any OpenAI-compatible endpoint) - those need no API key, just an `api_base`.


In [ ]:
!pip install nooa


## Setup

NOOA works with any LiteLLM-supported model - hosted or local. Pick one below. Replace `"your-api-key"` with a real key for hosted providers; local providers such as Ollama and vLLM do not need a key, just an `api_base`.


In [ ]:
from nooa.unifiedllm.registry import get_llm_client

# model = get_llm_client("claude-haiku-4-5", api_key="your-api-key")                         # Anthropic
model = get_llm_client("gpt-5.5", api_key="your-api-key")                                    # OpenAI
# model = get_llm_client("ollama_chat/qwen3:1.7b", api_base="http://localhost:11434")        # Ollama (local, no key)
# model = get_llm_client("hosted_vllm/Qwen/Qwen3-1.7B", api_base="http://localhost:8000/v1") # vLLM (local, no key)


## The Trip Request

We'll plan one relaxed Saturday in Lisbon. Keep your eye on the data moving between agents: the messy request goes into the scout, extracted preferences go into the itinerary builder, and the final markdown plan comes back to the parent.


In [ ]:
from nooa import Agent, print_prompt, strategy
from nooa.agentdoc import doc
from nooa.strategies import PredictStrategy


In [ ]:
TRAVELER_REQUEST = """
I'm spending one Saturday in Lisbon with my partner. We like great food, tilework,
bookshops, river views, and a little history, but we don't want a packed schedule.
Please avoid clubs and late nights. We are happy to walk, but not all day.
""".strip()

print(TRAVELER_REQUEST)


The itinerary builder will use a tiny hand-written city guide. In a real app this could be a database, an API result, or a dataframe. Here it is just enough live Python state for CodeAct to inspect through methods.


In [ ]:
LISBON_GUIDE = [
    {"name": "Manteigaria", "kind": "food", "area": "Chiado", "duration": 30, "tags": ["pastry", "classic", "quick"]},
    {"name": "Museu Nacional do Azulejo", "kind": "museum", "area": "Xabregas", "duration": 90, "tags": ["tilework", "history", "quiet"]},
    {"name": "Ler Devagar", "kind": "bookshop", "area": "LX Factory", "duration": 45, "tags": ["books", "design", "coffee"]},
    {"name": "Miradouro de Santa Catarina", "kind": "viewpoint", "area": "Bica", "duration": 35, "tags": ["river views", "sunset", "relaxed"]},
    {"name": "Time Out Market", "kind": "food", "area": "Cais do Sodre", "duration": 60, "tags": ["food hall", "easy", "busy"]},
    {"name": "MAAT river walk", "kind": "walk", "area": "Belem", "duration": 60, "tags": ["river views", "architecture", "gentle walk"]},
    {"name": "Alfama wander", "kind": "walk", "area": "Alfama", "duration": 75, "tags": ["history", "hills", "classic"]},
]

for place in LISBON_GUIDE:
    print(f"- {place['name']} ({place['kind']}, {place['area']}): {', '.join(place['tags'])}")


## Subagent 1: A Predict Scout

The first child has a narrow job: read the traveler request and return a short list of preferences. That is a good fit for `PredictStrategy`: one prompt, one validated Python value.

No custom Pydantic model here. The return type is just `list[str]`.


In [ ]:
class TripScoutAgent(Agent):
    """Extract travel preferences from a request."""

    @strategy(PredictStrategy())
    async def extract_preferences(self, request: str) -> list[str]:
        """Return 4 to 6 concise preferences or constraints directly supported by the request."""
        ...


## Subagent 2: A CodeAct Itinerary Builder

The second child writes the itinerary. We leave this method on the default strategy, so it uses CodeAct: the model gets a Python REPL and can call helper methods on `self`.

The helpers are intentionally ordinary Python. They give the CodeAct subagent a small surface for exploring the city guide without adding another schema or another framework concept.


In [ ]:
class ItineraryBuilderAgent(Agent):
    """Build relaxed city itineraries from preferences and a small city guide."""

    def __init__(self, guide: list[dict], **kwargs):
        super().__init__(**kwargs)
        self._guide = guide

    def find_by_tag(self, tag: str, n: int = 5) -> list[dict]:
        """Return places whose tags contain tag, case-insensitive."""
        q = tag.lower()
        matches = [place for place in self._guide if any(q in t.lower() for t in place["tags"])]
        return matches[:n]

    def relaxed_options(self, max_minutes: int = 75) -> list[dict]:
        """Return places that fit a relaxed pace."""
        return [place for place in self._guide if place["duration"] <= max_minutes]

    def format_stop(self, place: dict, when: str) -> str:
        """Format one itinerary stop as a markdown bullet."""
        return f"- **{when}: {place['name']}** ({place['area']}) — {place['kind']}, about {place['duration']} min"

    async def build_itinerary(self, preferences: list[str], guide_label: str) -> str:
        """Build a relaxed one-day itinerary.

        Use self.find_by_tag(), self.relaxed_options(), and self.format_stop() to inspect the guide.
        Return markdown with a title, 3 to 5 timed stops, and a short note explaining the pacing.
        """
        ...


## The Parent Orchestrator

Now compose the two children. `plan_day` has a real body, so the workflow order is fixed in Python:

1. Create the scout and await `extract_preferences`.
2. Create the itinerary builder and await `build_itinerary`.
3. Store the handoff on `self.last_preferences` so we can inspect it.

The child classes do **not** specify `llm=`. Because the parent creates them inside an active parent agent call, they inherit the parent's resolved model.

Notice that this method instantiates the child classes directly. The LLM is not deciding to spawn these subagents; the Python author is.


In [ ]:
class WeekendPlannerAgent(Agent, llm=model):
    """Coordinate subagents that turn a travel request into a one-day itinerary."""

    def __init__(self, guide: list[dict], **kwargs):
        super().__init__(**kwargs)
        self.guide = guide
        self.last_preferences: list[str] = []

    async def plan_day(self, request: str, city: str) -> str:
        """Run the subagent workflow and return a markdown itinerary."""
        scout = TripScoutAgent()
        preferences = await scout.extract_preferences(request)

        builder = ItineraryBuilderAgent(self.guide)
        itinerary = await builder.build_itinerary(preferences, guide_label=f"{city} city guide")

        self.last_preferences = preferences
        return itinerary


Run the workflow. From the outside, you call only the parent. The parent owns the child-agent wiring.


In [ ]:
planner = WeekendPlannerAgent(LISBON_GUIDE)

itinerary = await planner.plan_day(TRAVELER_REQUEST, city="Lisbon")

print(itinerary)


The intermediate handoff is still available. In larger systems this is usually where debugging starts: did the first child return the right small object for the next child?


In [ ]:
print("Preferences passed from scout to itinerary builder:")
for preference in planner.last_preferences:
    print("-", preference)


## Parallel Subagents

Sequential handoffs are not the only pattern. If a parent has several independent subtasks, it can spawn several child agents and run them with `asyncio.gather`.

The important rule is: **use one subagent instance per concurrent task.** Agentic methods have an internal per-instance lock, so concurrent calls on the same instance serialize. Separate child instances can run independently.


In [ ]:
import asyncio


class NeighborhoodScoutAgent(Agent):
    """Pitch one neighborhood for a traveler."""

    @strategy(PredictStrategy())
    async def pitch(self, neighborhood: str, preferences: list[str]) -> str:
        """Write one concise sentence explaining why the neighborhood fits the preferences."""
        ...


class ParallelWeekendPlannerAgent(WeekendPlannerAgent):
    """Weekend planner with a parallel neighborhood-scouting step."""

    async def compare_neighborhoods(self, neighborhoods: list[str]) -> list[str]:
        """Ask one scout per neighborhood to pitch options concurrently."""
        scouts = [NeighborhoodScoutAgent() for _ in neighborhoods]
        return await asyncio.gather(
            *(
                scout.pitch(neighborhood, self.last_preferences)
                for scout, neighborhood in zip(scouts, neighborhoods)
            )
        )


The parent below reuses the preferences extracted by the first workflow. Each neighborhood gets its own `NeighborhoodScoutAgent`, and all of their `pitch(...)` calls are awaited together.


In [ ]:
parallel_planner = ParallelWeekendPlannerAgent(LISBON_GUIDE)
parallel_planner.last_preferences = planner.last_preferences

neighborhood_pitches = await parallel_planner.compare_neighborhoods([
    "Chiado",
    "Belem",
    "Alfama",
])

for pitch in neighborhood_pitches:
    print("-", pitch)


This is still ordinary Python orchestration. The framework-specific part is just the child methods: each `pitch(...)` is an agentic method, and each child inherits the parent's model because it is created inside an active parent call.


## Letting CodeAct Spawn a Subagent

So far, the parent methods were deterministic Python: we wrote exactly which child agents to create. That is usually the clearest workflow design.

There is another pattern: a CodeAct method can decide to create and call a subagent from generated code. For that, the child class needs to be visible from `self`, so we expose it as a class attribute.

This is where the `TripScoutAgent = TripScoutAgent` pattern is useful. It is not a dataclass field; it is a class attribute alias that makes the child class part of the agent's model-facing surface.


In [ ]:
class AdaptiveWeekendPlannerAgent(Agent, llm=model):
    """Planner whose CodeAct method may create a scout subagent when useful."""

    TripScoutAgent = TripScoutAgent

    async def summarize_preferences(self, request: str) -> str:
        """Summarize the traveler's preferences.

        If the request has several constraints, create `self.TripScoutAgent()` and call
        `await scout.extract_preferences(request)`, then return a concise summary.
        """
        ...


Now `doc(...)` shows the child class on the parent. The generated CodeAct code can discover and instantiate it through `self.TripScoutAgent()`.


In [ ]:
adaptive_planner = AdaptiveWeekendPlannerAgent()
print(doc(adaptive_planner))


Run the adaptive method. This time the parent method itself is CodeAct, so the LLM can choose to create the subagent while solving the task.


In [ ]:
summary = await adaptive_planner.summarize_preferences(TRAVELER_REQUEST)
print(summary)


## What the Parent Exposes

`doc(planner)` renders the model-facing surface of an agent. The deterministic parent exposes its orchestration entrypoint and ordinary Python state. The adaptive parent above exposes a child class because its CodeAct method may need to instantiate it.


In [ ]:
print(doc(planner))


Each subagent has its own surface. If you instantiate a child directly in a notebook cell, pass `llm=model` explicitly because there is no active parent call to inherit from.


In [ ]:
builder = ItineraryBuilderAgent(LISBON_GUIDE, llm=model)
print(doc(builder))


### 🥷 Under the hood

`print_prompt` shows the exact prompt for one subagent call. This one is the CodeAct itinerary-builder prompt, so you should see the REPL instructions and the helper methods on `self`.

Notice what is not there: the original traveler request. The builder receives only the extracted preferences because the parent passed only those preferences.


In [ ]:
await print_prompt(
    builder.build_itinerary,
    preferences=[
        "Food and pastries are important.",
        "Tilework and history are priorities.",
        "Keep the day relaxed and avoid late nights.",
        "Include river views without walking all day.",
    ],
    guide_label="Lisbon city guide",
)


## LLM Inheritance and Overrides

This is the inheritance pattern used above:

```python
class ChildAgent(Agent):
    ...

class ParentAgent(Agent, llm=model):
    async def run(self):
        child = ChildAgent()       # inherits the parent's resolved LLM
        return await child.method()
```

If you create a child outside an active parent call, pass the model explicitly:

```python
child = ChildAgent(llm=model)
```

If one child should use a different model, override just that child in the parent:

```python
builder = ItineraryBuilderAgent(guide, llm=stronger_model)
```

Subagents also do not share memory, context blocks, or history automatically. Treat each handoff like a function call: pass the data the next child needs.


## Tracing the Handoff

If the trace viewer is not already running, start it in a terminal:

```bash
nooa start-dev
```

Then re-run `await planner.plan_day(...)` and inspect the trace. You should see a parent call with two child calls underneath it: a Predict scout, then a CodeAct itinerary builder. The useful thing to inspect is the boundary between them: a messy request becomes a short `list[str]`, and only that list is handed to the builder.


## Recap

- A subagent is just another `Agent` instance.
- The parent can be a pure Python orchestrator; it does not need an ellipsis body.
- Different child agents can use different strategies.
- Children can inherit the parent's LLM when created inside an active parent call.
- Subagents do not share state or history automatically. Pass data explicitly.

## Exercises

1. Add a third child that checks whether the itinerary violates any traveler constraints.
2. Give `ItineraryBuilderAgent` an explicit stronger model while leaving the scout on the parent model.
3. Add a deterministic helper on `ItineraryBuilderAgent` that groups places by area.
4. Print the scout prompt too and compare it with the CodeAct builder prompt.
